# Chapter 2b: Byte Pair Encoding

*Date: November 27, 2026*

**Description: Implement BPE tokenization.**

1. Load the text of “The Verdict” by Edith Wharton.
2. Use tiktoken's BPE algorithm to tokenize the text.
3. Create token embeddings.

Byte Pair Encoding provides a much more sophisticated tokenization scheme. Begin by installing the tiktoken library.

In [ ]:
pip install tiktoken

Import tiktoken and use its "gpt2" encode method.

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"\
    "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

Convert token IDs back into text using tiktoken's decode method.

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

Now, let's apply BPE to "The Verdict". Encode the raw text and print the encoded text length.

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

**Input-Target Pairs**

After encoding text, the next step is to generate input-target pairs. That is, the LLM uses the previous input tokens to predict the hidden target token.

*Example:*

```text
LLMs predict the next    word    from the past input.
          ︿               ︿              ︿
        input           target     remaining hidden
                                         text
```

Because the "window" of input expands as the LLM reads the text, we call this a "sliding window". We now implement a sliding window to 4 tokens from "The Verdict". The ```x``` variable contains the input tokens and the ```y``` variable contains the target tokens.

In [ ]:
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:  {x}")
print(f"y        {y}")

Iterating through these statements creates next-word prediction tasks.

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

In the output above, the arrows point from the input window to the current target. Repeating the code with the tokens converted to text allows us to visualize this better.

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

**PyTorch Tensors**

We then convert the input and target variables into PyTorch Tensors, which behave like multidimensional arrays.

*Example:*

*Sample text: To be or not to be, that is the question.*

```
x = tensor([["To",  "be",   "or",   "not"],
            ["to",  "be",   "that", "is"],
            ["the", "question"]])

y = tensor([["be",   "or",   "not", "to"],
            ["be",   "that", "is",  "the"],
            ["question"]])
```

We create a GPTDatasetV1 class, which tokenizes the text and uses a sliding window to chunk the text into overlapping sequences. It also contains methods for returning a row within the dataset and the total number of rows.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

ModuleNotFoundError: No module named 'torch'

Additionally create a Dataloader function to manage the Dataset class.

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Testing the dataloader on a batch size of 1 with context size 4; the input text is from "The Verdict".

In [26]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In the input above, the first tensor contains the inputs and the second tensor contains the targets.